In [ ]:
# Mount google drive to current runtime session to access required files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Import Libraries
import shutil
import pandas as pd
import numpy as np
import os
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
True
NVIDIA A100-SXM4-40GB


In [ ]:
# Import spectrogram zip to local storage for efficient loading

shutil.copy('/content/drive/MyDrive/spectrograms_np.zip',
            '/content/spectrograms_np.zip')

'/content/spectrograms_np.zip'

In [ ]:
# Unzip spectrogram file to extract all 150,000 npy files
!unzip -q spectrograms_np.zip -d /content/spectrograms_np

In [ ]:
# Size of spectrogram folder and number of npy files
!du -sh /content/spectrograms_np/*
!find /content/spectrograms_np -type f | wc -l

18G	/content/spectrograms_np/spectrograms_np
150017


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/lmd_full_metadata.csv')

# Filter out missing BPM
df = df[df['initial_bpm'].notna() & (df['initial_bpm'] != '')]

print(f"Total usable samples: {len(df)}")
print(f"Distribution of BPM:\n{df["initial_bpm"].describe()}")

Total usable samples: 155681
Distribution of BPM:
count    155681.000000
mean        115.538050
std          35.378194
min          20.000000
25%          92.000000
50%         116.000000
75%         130.000000
max         300.000000
Name: initial_bpm, dtype: float64


In [ ]:
df['npy_file'] = df['file'].str.replace("mid","npy")

# Build full file paths
data_dir = '/content/spectrograms_np/spectrograms_np/'
df['file_path'] = data_dir + df['npy_file']

In [ ]:
sample = df['file_path'].iloc[10]
print(f"Example path: {sample}")
print(f"File exists: {os.path.exists(sample)}")

Example path: /content/spectrograms_np/spectrograms_np/d5f8bcdd4a7dde087019def2de55f6c0.npy
File exists: True


In [ ]:
# Check which files actually exist
df['file_exists'] = df['file_path'].apply(os.path.exists)

print(f"Files found:   {df['file_exists'].sum()}")
print(f"Files missing: {(~df['file_exists']).sum()}")

# Keep only existing files
df = df[df['file_exists']]

from tqdm import tqdm

tqdm.pandas()  # enables progress_apply

df = df[df['file_path'].progress_apply(
    lambda p: np.load(p).shape == (128, 469)
)]

print(f"Remaining samples after removing different shapes: {len(df)}")

Files found:   150017
Files missing: 5664


100%|██████████| 150017/150017 [00:17<00:00, 8607.71it/s]


Remaining samples after removing different shapes: 149993


In [ ]:
sample = np.load('/content/spectrograms_np/spectrograms_np/01c9183f6c1af7e044d436d1cda0d56d.npy')
print(f"Shape: {sample.shape}")
print(f"Dtype: {sample.dtype}")
print(f"Min: {sample.min():.4f}, Max: {sample.max():.4f}")

Shape: (128, 469)
Dtype: float16
Min: -79.4375, Max: 0.0000


In [ ]:
class SpectrogramDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        x = np.load(self.file_paths[idx]).astype(np.float32)
        x = (x + 80) / 80                    # normalize to 0-1
        x = x[np.newaxis, :, :]              # add channel dim → (1, 128, 469)
        return torch.tensor(x), torch.tensor(self.labels[idx], dtype=torch.float32)

In [ ]:
# Normalize BPM for model
bpm_mean = df['initial_bpm'].mean()
bpm_std  = df['initial_bpm'].std()
df['bpm_normalized'] = (df['initial_bpm'] - bpm_mean) / bpm_std

file_paths = df['file_path'].tolist()
regression = df['bpm_normalized'].tolist()

# Split into train/val

X_train, X_val, y_train, y_val = train_test_split(
    file_paths, regression,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_val, y_val,
    test_size=0.5,
    random_state=42
)

print(f"Train samples: {len(X_train)}")
print(f"Val samples:   {len(X_val)}")
print(f"Test samples:   {len(X_test)}")

train_dataset = SpectrogramDataset(X_train, y_train)
val_dataset   = SpectrogramDataset(X_val,   y_val)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,
                          num_workers=8, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False,
                          num_workers=8, pin_memory=True)

Train samples: 119994
Val samples:   14999
Test samples:   15000


In [ ]:
import torch.nn as nn

class SpectrogramCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.features = nn.Sequential(
        nn.Conv2d(1,32,kernel_size=3,padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32, 64, 3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64, 128, kernel_size = 3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(128, 256, kernel_size = 3, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(256, 512, kernel_size = 3, padding=1),
        nn.BatchNorm2d(512),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

      )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(512*4*14, 512),
        nn.ReLU(),
        nn.Dropout(.3),
        nn.Linear(512,256),
        nn.ReLU(),
        nn.Dropout(.3),
        nn.Linear(256, 1)
    )


  def forward(self, x):
      x = self.features(x)
      x = self.classifier(x)
      return x

model = SpectrogramCNN().cuda()
print(model)

SpectrogramCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=

In [ ]:
# Early stopping class only using validation values

class EarlyStopper:
    def __init__(self, patience=4, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss - self.min_delta:
            self.min_validation_loss = validation_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [ ]:
import torch.optim as optim
from torch.amp import GradScaler, autocast
import time

start = time.time()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)
scaler = GradScaler('cuda')
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.7)
early_stopper = EarlyStopper(patience=4, min_delta=.25)

CHECKPOINT_PATH = '/content/drive/MyDrive/cnn_project/checkpoints/checkpoint_regression.pth'

for epoch in range(25):
    # --- Training ---
    model.train()
    running_loss = 0.0
    running_mae  = 0.0

    for images, labels in train_loader:
        images, labels = images.cuda(), labels.cuda()
        optimizer.zero_grad()

        with autocast('cuda'):
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        running_mae  += torch.abs(outputs - labels).mean().item()

    train_mae     = running_mae / len(train_loader)
    train_mae_bpm = train_mae * bpm_std

    # --- Validation ---
    model.eval()
    val_mae = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.cuda(), labels.cuda()
            outputs = model(images).squeeze()
            val_mae += torch.abs(outputs - labels).mean().item()

    val_mae     = val_mae / len(val_loader)
    val_mae_bpm = val_mae * bpm_std

    # --- Checkpoint ---
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, CHECKPOINT_PATH)

    # --- Save best model separately ---
    if val_mae_bpm < early_stopper.min_validation_loss:
        torch.save(model.state_dict(), '/content/drive/MyDrive/cnn_project/checkpoints/best_model_spect_regress_CNN.pth')

    print(f"Epoch {epoch+1:2d}/25 | Loss: {running_loss/len(train_loader):.4f} | Train MAE: {train_mae_bpm:.1f} BPM | Val MAE: {val_mae_bpm:.1f} BPM")
    scheduler.step()
    print(f"LR: {scheduler.get_last_lr()[0]:.6f}")

    if early_stopper.early_stop(val_mae_bpm):
      print(f"EARLY STOPPING AT EPOCH {epoch+1:2d}, best val: {early_stopper.min_validation_loss:.1f} BPM")
      break

end = time.time()
print(f"Total training time: {(end-start)/60:.1f} minutes")

Epoch  1/25 | Loss: 1.2920 | Train MAE: 26.0 BPM | Val MAE: 22.1 BPM
LR: 0.000300
Epoch  2/25 | Loss: 0.7341 | Train MAE: 20.7 BPM | Val MAE: 23.2 BPM
LR: 0.000300
Epoch  3/25 | Loss: 0.6674 | Train MAE: 19.1 BPM | Val MAE: 20.5 BPM
LR: 0.000300
Epoch  4/25 | Loss: 0.6317 | Train MAE: 18.2 BPM | Val MAE: 20.1 BPM
LR: 0.000300
Epoch  5/25 | Loss: 0.6006 | Train MAE: 17.6 BPM | Val MAE: 19.5 BPM
LR: 0.000300
Epoch  6/25 | Loss: 0.5712 | Train MAE: 17.0 BPM | Val MAE: 19.0 BPM
LR: 0.000300
Epoch  7/25 | Loss: 0.5431 | Train MAE: 16.3 BPM | Val MAE: 18.5 BPM
LR: 0.000210
Epoch  8/25 | Loss: 0.4865 | Train MAE: 15.3 BPM | Val MAE: 17.8 BPM
LR: 0.000210
Epoch  9/25 | Loss: 0.4374 | Train MAE: 14.4 BPM | Val MAE: 17.3 BPM
LR: 0.000210
Epoch 10/25 | Loss: 0.3884 | Train MAE: 13.5 BPM | Val MAE: 17.2 BPM
LR: 0.000210
Epoch 11/25 | Loss: 0.3378 | Train MAE: 12.6 BPM | Val MAE: 15.9 BPM
LR: 0.000210
Epoch 12/25 | Loss: 0.2809 | Train MAE: 11.5 BPM | Val MAE: 16.0 BPM
LR: 0.000210
Epoch 13/25 | Lo

In [ ]:
# Evaluating model against test

model = SpectrogramCNN().cuda()
model.load_state_dict(torch.load('/content/drive/MyDrive/cnn_project/checkpoints/best_model_spect_regress_CNN.pth'))
model.eval()

# Dataloader for test data
test_dataset = SpectrogramDataset(X_test, y_test)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

test_mae = 0.0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.cuda(), labels.cuda()
        outputs = model(images).squeeze()
        test_mae += torch.abs(outputs - labels).mean().item()

test_mae     = test_mae / len(test_loader)
test_mae_bpm = test_mae * bpm_std


print(f"Test BPM MAE: {test_mae_bpm:.1f}")

Test BPM MAE: 13.6
